# Week 7.2 — Tiny pricer (baseline + optional QLoRA)

Same exercise as Week 7: predict **price from description** using the shared `pricer` stack (`Item`, `evaluate`), but with a **smaller base model** and **fewer moving parts** than typical Qwen/Llama-heavy notebooks.

| Piece | Choice here |
|-------|----------------|
| Base model | `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (ungated, fast to load) |
| Data | `ed-donner/items_lite` (flip `LITE_MODE` for full) |
| Training | Optional QLoRA via `SFTTrainer`; run on **CUDA** or course Colab |

**Tip:** On Apple Silicon / CPU, use a small `EVAL_SIZE` and `workers=1` in `evaluate` so inference stays stable.

In [ ]:
import os
import re
import sys

week7_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, week7_root)

from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
if os.environ.get("HF_TOKEN"):
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)

# --- knobs ---
LITE_MODE = True
USERNAME = "ed-donner"
DATASET = f"{USERNAME}/items_lite" if LITE_MODE else f"{USERNAME}/items_full"
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
SUMMARY_TOKEN_CAP = 110
EVAL_SIZE = 30

train, val, test = Item.from_hub(DATASET)
for it in train + val + test:
    if not it.summary:
        it.summary = it.title

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

for it in tqdm(train + val, desc="prompts (train+val)"):
    it.make_prompts(tokenizer, SUMMARY_TOKEN_CAP, do_round=True)
for it in tqdm(test, desc="prompts (test)"):
    it.make_prompts(tokenizer, SUMMARY_TOKEN_CAP, do_round=False)

print(f"Train {len(train):,} | Val {len(val):,} | Test {len(test):,} | model {BASE_MODEL}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig


def parse_price(text: str) -> float:
    text = (text or "").replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", text)
    return float(m.group()) if m else 0.0


def load_causal_lm():
    if torch.cuda.is_available():
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=bnb,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        dtype = torch.float16 if torch.backends.mps.is_available() else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, torch_dtype=dtype, trust_remote_code=True
        )
        dev = "mps" if torch.backends.mps.is_available() else "cpu"
        model.to(dev)
    model.eval()
    return model


_base = None


def get_base_model():
    global _base
    if _base is None:
        _base = load_causal_lm()
    return _base


def price_from_model(model, item: Item) -> str:
    prompt = item.test_prompt()
    dev = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1] :]
    tail = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return str(parse_price(tail))


def baseline_pricer(item: Item) -> str:
    return price_from_model(get_base_model(), item)


eval_workers = 5 if torch.cuda.is_available() else 1
evaluate(
    baseline_pricer,
    test,
    size=min(EVAL_SIZE, len(test)),
    workers=eval_workers,
)

## Optional: QLoRA (CUDA)

Install if needed: `pip install peft bitsandbytes trl accelerate datasets`.

Set `RUN_QLORA = True` on a GPU machine or use the course Colab, then point `ADAPTER_PATH` at the saved folder for the last cell.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

RUN_QLORA = False
MAX_SAMPLES = 800
OUTPUT_DIR = os.path.join(week7_root, "community_contributions", "sammyloto", "qlora_tinyllama_pricer")
NUM_EPOCHS = 1
LR = 2e-4

if RUN_QLORA and torch.cuda.is_available():
    from datasets import Dataset
    from peft import LoraConfig
    from trl import SFTConfig, SFTTrainer

    def to_sft_text(batch):
        return {"text": [p + c for p, c in zip(batch["prompt"], batch["completion"])]}

    rows = [{"prompt": it.prompt, "completion": it.completion} for it in train[:MAX_SAMPLES]]
    ds = Dataset.from_list(rows).map(to_sft_text, batched=True)

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    ft_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb,
        device_map="auto",
        trust_remote_code=True,
    )
    lora = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=LR,
        logging_steps=20,
        max_seq_length=512,
        dataset_text_field="text",
    )
    trainer = SFTTrainer(
        model=ft_model,
        train_dataset=ds,
        peft_config=lora,
        args=args,
        tokenizer=tokenizer,
    )
    trainer.train()
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Saved adapter to", OUTPUT_DIR)
else:
    print("Skipping QLoRA (set RUN_QLORA=True on CUDA, or train in Colab).")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

ADAPTER_PATH = None  # e.g. OUTPUT_DIR after training

_peft = None


def get_peft_model():
    global _peft
    if _peft is not None:
        return _peft
    if not ADAPTER_PATH:
        return None
    from peft import PeftModel

    if torch.cuda.is_available():
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=bnb,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        dtype = torch.float16 if torch.backends.mps.is_available() else torch.float32
        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, torch_dtype=dtype, trust_remote_code=True
        )
        base.to("mps" if torch.backends.mps.is_available() else "cpu")
    _peft = PeftModel.from_pretrained(base, ADAPTER_PATH)
    _peft.eval()
    return _peft


def qlora_pricer(item: Item) -> str:
    m = get_peft_model()
    if m is None:
        return "0"
    return price_from_model(m, item)


eval_workers = 5 if torch.cuda.is_available() else 1

if ADAPTER_PATH:
    evaluate(
        qlora_pricer,
        test,
        size=min(EVAL_SIZE, len(test)),
        workers=eval_workers,
    )
else:
    print("Set ADAPTER_PATH to evaluate the adapter.")